# Field maps

A field map is an external field on a grid: a magnet or coil map, or a field computed offline on a fitted mesh. It uses the coordinates, names and units of the output format (`b_field_<i>` in T, `e_field_<i>` in V/m on `x`, `y`, `z` in m), so a map written by one code is read by any other. `FieldMap` builds, checks, stores and interpolates them.

In [ ]:
import numpy as np
import plasma_plots
import xarray as xr

from takeout import FieldMap, validate_field_map

plasma_plots.set_backend("plotly")
plasma_plots.set_math("latex")

## A map from a function

A magnetic mirror along `x`, sampled on 41 nodes: $B_x(x) = B_0\,(1 + (R - 1)\,(2x/L - 1)^2)$ with the mirror ratio $R = 3$. `from_function` takes the field as a function of positions `(N, ndim)` returning `(N, num_components)`, the signature of the field functions of the codes.

In [ ]:
length, b0, ratio = 0.1, 0.05, 3.0


def mirror(points):
    s = 2.0 * points[:, 0] / length - 1.0
    bx = b0 * (1.0 + (ratio - 1.0) * s**2)
    return np.stack([bx, np.zeros_like(bx), np.zeros_like(bx)], axis=1)


field_map = FieldMap.from_function(
    mirror,
    {"x": np.linspace(0.0, length, 41)},
    kind="magnetic",
    source="analytic mirror",
)
field_map.dataset

The map has one dim, so it is uniform along `y` and `z`: it can be evaluated at 3D positions, and only their `x` is used. Between the nodes the interpolation is multilinear, second order in the node spacing:

In [ ]:
points = np.array([[0.0123, 0.4, -1.0], [0.05, 0.0, 0.0], [0.0871, 0.1, 0.2]])
error = field_map.magnetic_field(points)[:, 0] - mirror(points)[:, 0]
np.abs(error) / b0

## Outside the map

Positions beyond the last node take the field of the nearest boundary node (`outside="clip"`, the default), zero (`"zero"`), or raise (`"error"`): a map that does not cover the domain of a run is a choice, not an accident.

In [ ]:
beyond = np.array([[-0.02], [0.05], [0.12]])
for outside in ("clip", "zero"):
    values = FieldMap(field_map.dataset, outside=outside).magnetic_field(beyond)[:, 0]
    print(f"{outside:>5}: {values}")
try:
    FieldMap(field_map.dataset, outside="error").magnetic_field(beyond)
except ValueError as error:
    print(error)

The map against the function on a finer grid and beyond its ends, with the clipped field outside:

In [ ]:
x = np.linspace(-0.02, 0.12, 281)
comparison = (
    xr.Dataset(
        {
            "map": ("x", field_map.magnetic_field(x[:, None])[:, 0]),
            "function": ("x", mirror(x[:, None])[:, 0]),
        },
        coords={"x": ("x", x, {"units": "m", "long_name": "$x$"})},
    )
    .to_dataarray("source")
    .assign_attrs(units="T", long_name="$B_x$")
)
with plasma_plots.figure() as fig:
    comparison.plasma.plot.profiles(x="x", over="source", ax=fig[0])

## Files and other codes

`write` stores the map as netCDF with its attributes; `read` checks it with `validate_field_map`, which lists every problem, as the output format's validator does. A map written with xarray alone in the same layout is read the same way.

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as folder:
    path = field_map.write(Path(folder) / "mirror.nc")
    back = FieldMap.read(path)
back.dims, back.components("magnetic"), back.dataset.attrs

In [ ]:
wrong = field_map.dataset.assign_attrs(frame="polar")
wrong["b_field_x"] = wrong["b_field_x"].assign_attrs(units="G")
validate_field_map(wrong)